In [1]:
# This notebook is a replication of Gu, Kelly, Xiu (2020) Emprical Asset Pricing using Machine Learning;
# In the orginal paper, the authors used 60 years of data from 1957-2016 to evaluate machine learning models;
# The last 30 years (1987-2016) were used for testing;
# Here, the testing period is extended by 5 years, i.e 35-year predictions from 1987-2021;

# To prepare data for training, validation, and testing, this notebook does the following:
## 1. Ranking stock-level characteristics and mapping them to [-1,1];
## 2. Computing interaction terms between characteristics and marco-economic predictors;
## 3. Computing excess returns;
## 4. Splitting the whole sample to containing training, validation, and testing samples 35 times.
### Each time the training window is increased by 1 year while the validation window is fixed with 12 years.

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import os, psutil
import os.path as path
import gc

# Stock-month lagged characteristics; url = https://dachxiu.chicagobooth.edu/download/datashare.zip
char = pd.read_csv('datashare.csv')
char['mdate'] = pd.to_datetime(char['DATE'].astype(str)).dt.to_period('M') #create MM-YYYY datetime
char = char.sort_values(by=['mdate', 'permno'])  #ensure stock-month order
start, end = char['mdate'].min(), char['mdate'].max()
print(f'Characteristics data:')
print(f'Shape: {char.shape}')
print(f'Start date: {start}, End date: {end}')
print(char.head(10))
print(f'-------------------------------------------------------------------------------------------------------------------------------------------------------')
char_variables = [col for col in char.columns if col not in ['permno', 'DATE', 'mdate', 'sic2']]  #K=94 characteristics
print(f'Number of stock characteristics: {len(char_variables)}')
print(f'Stock characteristics:')
print(char_variables)
#check discrete characteristics
print(f'Checking discrete characteristics:')
char_discrete = []
for c in char_variables:
    if char[c].nunique() <= 2:
        char_discrete.append(c)
        print(f'{c}: {char[c].nunique()} unique values: {char[c].unique()}')
char_cont = [c for c in char_variables if c not in char_discrete]
print(f'-------------------------------------------------------------------------------------------------------------------------------------------------------')
#Monthly median values
median = char.groupby('mdate')[char_cont].median().reset_index(drop=False)
for c in char_cont:
    name = f'{c}_med'
    median = median.rename(columns={c: name})
median = median.fillna(0) #for months when all stocks have NaN values, set median=0

#for continuous characteristics: fill nans with monthly median; for discrete: fill nans with 0
print(f'Processing NA values:')
print(f'For discrete characteristics, NA values will be set to 0;')
print(f'For continuous characteristics, NA values will set to monthly median value.')
for c in char_variables:
    na_idx = char[char[c].isna()].index
    print(f'NA counts for {c} before: {char[c].isna().sum()}')
    if c in char_discrete:
        char[c] = char[c].fillna(0)
    else:
        for d in char.loc[na_idx]['mdate'].unique():
            char.loc[(char[c].isna()) & (char['mdate']==d), c] = median.loc[(median['mdate']==d), f'{c}_med'].item()
    print(f'NA counts for {c} after: {char[c].isna().sum()}')
print(f'-------------------------------------------------------------------------------------------------------------------------------------------------------')
# Download stocks monthly returns from WRDS
# stocklist = np.sort(char['permno'].unique())
# stock_str = ','.join(str(x) for x in stocklist)
# mret = conn.raw_sql(f"""select permco, permno, date, prc, ret, retx, shrout
#                        from crsp.msf
#                        where permno in ({stock_str})
#                        and date>='01/01/1957'
#                        """,
#                     date_cols=['date'])
#mret.to_csv('mret.csv', index=False)
mret = pd.read_csv('mret.csv')
mret['mdate'] = pd.to_datetime(mret['date'].astype(str)).dt.to_period('M')

#align stock-month characteristics and returns
char = pd.merge(char, mret[['mdate','permno','ret']], on=['mdate','permno'])
na_ret = char[char['ret'].isna()].index
char = char.drop(index=na_ret)
char = char.reset_index(drop=True)

#Rank stocks' characteristics each month and map to [-1,1]
def _rank(df, group_by_col, rank_col):
    df = df.copy()
    df['rank'] = df.groupby(group_by_col)[rank_col].rank(method='min', ascending=True)
    max_rank = pd.DataFrame({
        'max_rank': df.groupby(group_by_col)['rank'].max()
    }).reset_index(drop=False)
    df = pd.merge(df, max_rank, on=group_by_col)
    df['rank_map'] = np.where(
        df['max_rank'] - 1 > 0,
        2 * (df['rank'] - 1) / (df['max_rank'] - 1) - 1,
        0.0
    )
    rank_df = df[['mdate', 'permno', 'rank_map']].copy()
    rank_df.rename(columns={'rank_map': rank_col}, inplace=True)
    return rank_df
for c in char_variables:
    if c == char_variables[0]:
        df = char[['mdate', 'permno', c]]
        char_rank = _rank(df, group_by_col='mdate', rank_col=c)
    elif c in char_discrete:
        char_rank = pd.merge(char_rank, char[['mdate', 'permno', c]], on=['mdate', 'permno'])
    else:
        df = char[['mdate', 'permno', c]]
        rank_df = _rank(df, group_by_col='mdate', rank_col=c)
        char_rank = pd.merge(char_rank, rank_df, on=['mdate', 'permno'])
print(f'Characteristics after ranking and mapping to [-1,1]:')
print(f'Check if any characteristics is outside [-1,1]:')
lower = char_rank.groupby('mdate')[char_variables].min()
upper = char_rank.groupby('mdate')[char_variables].max()
print(f'Less than -1:{np.where(lower < -1)}')
print(f'Greater than 1:{np.where(upper > 1)}')
print(char_rank.head(10))
print(f'-------------------------------------------------------------------------------------------------------------------------------------------------------')

# Macro predictors data
macros_raw = pd.read_csv(
    f'https://docs.google.com/spreadsheets/d/10_nkOkJPvq4eZgNl-1ys63PzhbnM3S2y/export?format=csv&gid=1922816101')
## Compute 8 macro predictors
macros_raw['mdate'] = pd.to_datetime(macros_raw['yyyymm'].astype(str), format='%Y%m').dt.to_period('M')
macro = macros_raw.copy()

### dividend price ratio = log(dividends/lagged prices) of S&P500
macro['dp'] = np.log(macro['d12'] / macro['price'].shift(1))

### earnings price ratio = log(earnings/ lagged prices) of S&P500
macro['e/p'] = np.log(macro['e12'] / macro['price'].shift(1))
### term spread = long term yield on gov bond - treasury bill
macro['tms'] = macro['lty'] - macro['tbl']

### default yield spread = BAA - AAA corp bond yields
macro['dfy'] = macro['BAA'] - macro['AAA']

macro = macro[['mdate', 'dp', 'e/p', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']]
macro_variables = ['dp', 'e/p', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']

## shift to create lagged predictors
macro[macro_variables] = macro[macro_variables].shift(1)
macro = macro.drop(index=macro[(macro['mdate'] < '1957-01')].index)
macro = macro.reset_index(drop=True)

print(f'Macro-economic predictors data:')
print(f'Number of macro predictors: {len(macro_variables)}')
print(macro_variables)
print(macro.head())
print(f'-------------------------------------------------------------------------------------------------------------------------------------------------------')
# Compute excess returns
char = pd.merge(char, macros_raw[['mdate','Rfree']], on=['mdate'])
char['excessret'] = char['ret'] - char['Rfree']

# Merge excess returns, ranked characteristics, sic dummies & macros to 1 df
X = pd.merge(char_rank, char[['mdate','permno','sic2','excessret']], on=['mdate','permno'])
X = pd.merge(X, macro, on='mdate')

# 74 sic dummies
sic_categories = [np.sort(X['sic2'].dropna().unique())]
sic_ohe = OneHotEncoder(handle_unknown='ignore', categories=sic_categories, sparse_output=False)

# Checking total number of signals
n_interactions = len(macro_variables) * len(char_variables)
n_signals = len(char_variables) + n_interactions + sic_categories[0].shape[0]
print(f'Interaction terms: {n_interactions}')
print(f'Sic dummies: {sic_categories[0].shape[0]}')
print(f'Signals: {n_signals}')
print(f'-------------------------------------------------------------------------------------------------------------------------------------------------------')

#Create stopping indices for training, validation & testing samples
trainperiod = pd.to_datetime(pd.date_range(start='1974-12-31', end='2008-12-31', freq='YE')).to_period('M')
idx = []
for j in trainperiod:
    train = X[X['mdate'] <= j].shape[0]  #training samples; first training sample = 18 years; recursively increase by 1 year
    val = train + X[X['mdate'].between(j + 1, j + 12*12)].shape[0]  #validation samples = 12 years after training sample
    test = val + X[X['mdate'].between(j + 12*12 + 1, j + 13*12)].shape[0]  #testing samples = 1 year after validation sample
    idx.append((train, val, test))
idx = pd.DataFrame(idx, columns=['train', 'val', 'test'])
idx.to_csv('idx.csv', index=False)
print(f'Sample split:')
print(f'Each subsample indices are defined based on the X df above, which stores both target and feature variables')
print(f'For e.g: the first split has training sample indices go from 0 to 478008, validation sample from 478009 to 1248578, testing sample from 1248579 to 1331524')
print(idx)
print(f'-------------------------------------------------------------------------------------------------------------------------------------------------------')

# For easier computation at later stages, I also extract the following:
idx = idx.to_numpy()
char_test = char.iloc[idx[0][1]:].reset_index(drop=True)

## testing sample monthly periods
oos_periods = np.array(char_test['mdate'])

## testing sample stock id
permno = np.array(char_test['permno'])

## get testing sample market values
mktval = np.array(char_test['mvel1'])

## get indices of the top 1000 and bottom 1000 stocks by market value from testing sample
mvel_rank = char_test.groupby('mdate')['mvel1'].rank(method='min',ascending=True)
mvel_max_rank = mvel_rank.groupby(char_test['mdate']).transform('max')
top1000 = mvel_rank[mvel_rank > mvel_max_rank - 1000].index
bot1000 = mvel_rank[mvel_rank < 1000].index

## s&p500 log returns for benchmark
sp500 = macros_raw[['mdate','price']]
sp500['logret'] = np.log(sp500['price']/sp500['price'].shift(1))
sp500ret = sp500[sp500['mdate'].between('1987-01','2021-12')]['logret']

# Compute interaction terms--------------------------------------------------------------------------------------------------------------------------------------
def _interaction(df):
    chars = df[char_variables].to_numpy(dtype='float32')
    macros = df[macro_variables].to_numpy(dtype='float32')
    z = [chars * macros[:, [i]] for i in range(macros.shape[1])]
    z_stacked = np.hstack(z)
    sic = sic_ohe.fit_transform(df[['sic2']])
    z_stacked = np.hstack((z_stacked, sic))
    gc.collect()
    return np.hstack((df[char_variables],z_stacked))

# Save design matrix and target variables as memmap files
Xdata = np.memmap(path.join(os.getcwd(), 'Xdata.dat'), dtype='float32', mode='w+', shape=(X.shape[0], n_signals))
ydata = np.memmap(path.join(os.getcwd(), 'ydata.dat'), dtype='float32', mode='w+', shape=(X.shape[0], 1))

for i in range(0, X.shape[0], 10000):
    end = min(i + 10000, X.shape[0])
    X_df = X.iloc[i:end]
    z = _interaction(X_df)
    Xdata[i:end] = z[:]

ydata[:] = X[['excessret']][:]

# Save other files
np.save('top1000.npy', top1000)
np.save('bot1000.npy', bot1000)
np.save('permno.npy', permno)
np.save('mvel.npy', mktval)
np.save('oos_periods.npy', oos_periods)
np.save('char_variables.npy', char_variables)
np.save('macro_variables.npy', macro_variables)
np.save('sp500ret.npy', sp500ret)

Characteristics data:
Shape: (4117300, 98)
Start date: 1957-01, End date: 2021-12
   permno      DATE       mvel1      beta    betasq     chmom     dolvol  \
0   10006  19570131   82249.000  1.122846  1.260784  0.047180   9.569953   
1   10014  19570131    3903.375  0.426734  0.182102 -0.275641   6.237836   
2   10022  19570131    9273.250  1.066449  1.137313 -0.025490   7.008844   
3   10030  19570131   54465.875  0.926038  0.857547  0.018171   9.825337   
4   10057  19570131   40250.000  1.247748  1.556875  0.025785   7.901007   
5   10065  19570131   76945.250  1.264106  1.597964  0.193374   8.906529   
6   10102  19570131  186193.500  1.445027  2.088104 -0.058910  10.375318   
7   10137  19570131  225984.000  0.585546  0.342864 -0.090805   8.887653   
8   10145  19570131  962994.375  1.132210  1.281900 -0.084218  10.850652   
9   10153  19570131  279846.875  1.173314  1.376666 -0.008426  10.547950   

    idiovol    indmom     mom1m  ...  ms  baspread           ill    maxret  \
0  